## Centerline

In [ ]:
import numpy
import sys
from scipy.ndimage import distance_transform_edt
from skimage.feature import peak_local_max
from scipy.spatial import cKDTree
import vtk
from sklearn.decomposition import PCA
from scipy.interpolate import UnivariateSpline
import meshio

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import export_centerline, export_mask, set_the_offset

base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/re-meshed_alt-1_h_1-0/"



mesh_folder = "INP_re-meshed_h_1-0_tolerance_1-5_all/"

out_folder = "centerlines/"

h = 1.0

name = "lat"
id = 4

In [12]:
def fit_mesh_centerline_simple(
    mesh_points, 
    nbins=15, 
    trim=0.05, 
    smoothness=2.0, 
    spline_order=3, 
    n_eval_points=200,
    target_point=None
):

    points = numpy.asarray(mesh_points, dtype=float)
    if len(points) < 10:
        raise ValueError("Not enough points in mesh to fit centerline.")

    # 1. Compute principal axis using PCA
    pca = PCA(n_components=3)
    pca.fit(points)

    axis = pca.components_[0]      # Primary axis direction
    origin = pca.mean_             # Centroid of point cloud

    # Parameterize coordinates along the primary axis
    t = (points - origin) @ axis

    # Optional orientation check: force t to increase toward target_point (e.g., eyeball)
    if target_point is not None:
        direction_to_target = numpy.array(target_point) - origin
        if numpy.dot(axis, direction_to_target) < 0:
            axis = -axis
            t = -t

    order_idx = numpy.argsort(t)
    t_sorted = t[order_idx]

    # 2. Remove ends (trimming)
    tmin, tmax = t.min(), t.max()
    t_lo = tmin + trim * (tmax - tmin)
    t_hi = tmax - trim * (tmax - tmin)

    core_mask = (t >= t_lo) & (t <= t_hi)
    points_core = points[core_mask]
    t_core = t[core_mask]

    # 3. Compute cross-section centers
    bins = numpy.linspace(t_lo, t_hi, nbins)
    centers = []
    t_centers = []

    for i in range(nbins - 1):
        m = (t_core >= bins[i]) & (t_core < bins[i + 1])
        if numpy.sum(m) > 0:
            centers.append(points_core[m].mean(axis=0))  # centroid of slice
            t_centers.append(t_core[m].mean())           # average axial position

    centers = numpy.array(centers)
    t_centers = numpy.array(t_centers)

    if len(t_centers) <= spline_order:
        raise ValueError(f"Too few valid bins ({len(t_centers)}) for spline order {spline_order}.")

    # 4. Fit parametric centerline (UnivariateSpline for x, y, z)
    sx = UnivariateSpline(t_centers, centers[:, 0], k=spline_order, s=smoothness)
    sy = UnivariateSpline(t_centers, centers[:, 1], k=spline_order, s=smoothness)
    sz = UnivariateSpline(t_centers, centers[:, 2], k=spline_order, s=smoothness)

    # 5. Evaluate final centerline curve across full range of t
    tt = numpy.linspace(t_sorted.min(), t_sorted.max(), n_eval_points)
    centerline = numpy.vstack([
        sx(tt),
        sy(tt),
        sz(tt)
    ]).T

    splines = {'sx': sx, 'sy': sy, 'sz': sz}
    
    return centerline, splines

In [13]:
# 1. Read input mesh
# filename = name + "_h"+str(h).replace(".","_")+"_extended_V2.inp"  # your .inp mesh file

filename = name + "_h"+str(h).replace(".","_")+"_extended.inp"  # your .inp mesh file

mesh = meshio.read(base + mesh_folder + filename)


# Extract 3D node coordinates directly (X, Y, Z)
mesh_points = mesh.points

# 2. Fit centerline
centerline, splines = fit_mesh_centerline_simple(
    mesh_points=mesh_points,
    nbins=30,
    trim=0.005,
    smoothness=2,
    spline_order=3
)

# 3. Export parametric spline coefficients (.npz)
sx, sy, sz = splines['sx'], splines['sy'], splines['sz']
numpy.savez(
    base + out_folder + "centerline_id_"+str(id)+".npz",
    tx=sx._eval_args[0], cx=sx._eval_args[1], kx=sx._eval_args[2],
    ty=sy._eval_args[0], cy=sy._eval_args[1], ky=sy._eval_args[2],
    tz=sz._eval_args[0], cz=sz._eval_args[1], kz=sz._eval_args[2]
)

# 4. Export centerline to .vtp
export_centerline(
    centerline, 
    1, 
    base + out_folder + name + "centerline_id_"+str(id)+".vtp", 
    [0,0,0]
)




ReadError: File /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/re-meshed_alt-1_h_1-0/INP_re-meshed_h_1-0_tolerance_1-5_all/INP_re-meshed_h_1-0_tolerance_1-5_all/lat_h1_0_extended.inp not found.

In [ ]:
# 4. Compute centerline length (arc length via piecewise-linear approximation)
segment_lengths = numpy.linalg.norm(numpy.diff(centerline, axis=0), axis=1)
centerline_length = segment_lengths.sum()

print(f"Centerline length: {centerline_length:.4f}")

Centerline length: 32.6567
